# Source-Aware Evaluation of AraBERT for Arabic Fake News Detection
**CAI 421 Deep Learning - Checkpoint 2 code** (AFND, binary credibility: credible vs. not credible)

This notebook:
1. Downloads AFND from Kaggle, maps source labels, removes undecided sources, cleans and de-duplicates the data, and samples a fixed number of articles per source.

2. Evaluates on two splits of the same sample: a random article-level split and a source-disjoint split (repeated over random source assignments).

3. Trains two baselines: TF-IDF + linear SVM, and default AraBERT fine-tuning.

4. Runs the LLRD / class-weighted-loss ablation and the SHAP + feature-weight analysis.


To run: set the runtime to T4 GPU and Run all. Use SMOKE = True first for a quick check. Results are saved in results/.


In [ ]:
!pip -q install transformers kagglehub shap

In [ ]:
import os, re, json, glob, random, hashlib, time, copy, math
import numpy as np, pandas as pd
import torch, torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score, recall_score

# ----------------------------- CONFIG (edit here only) -----------------------------
SMOKE = False            # True = tiny run to test the pipeline (minutes)
SEED = 42                # base seed; repeat r uses SEED + r
CFG = dict(
    n_per_source = 60 if SMOKE else 300,   # per-source cap -> ~103 sources x 300 = ~30k articles
    min_chars    = 200,                    # drop very short articles
    max_chars    = 2000,                   # text kept per article (title + body prefix)
    split        = (0.70, 0.15, 0.15),     # train / val / test
    n_repeats    = 2 if SMOKE else 3,      # random-source-assignment repeats (SVM)
    bert_repeats = 1,                      # repeats for AraBERT (raise to 3 when time allows)
    model_name   = "aubmindlab/bert-base-arabertv02",
    max_len      = 128,
    batch_size   = 32,
    epochs       = 1 if SMOKE else 2,
    lr_top       = 2e-5,                   # eta_L in Eq. (1)
    llrd_xi      = 0.9,                    # xi in Eq. (1); 1.0 = uniform LR (default fine-tuning)
    warmup       = 0.1,
    weight_decay = 0.01,
)
RUN_ABLATION = False     # LLRD / weighted-loss ablation (run after baselines look right)
RUN_ANALYSIS = False     # SHAP / feature-weight analysis
# ------------------------------------------------------------------------------------

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("results", exist_ok=True)
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed(SEED)
print("device:", DEVICE, "| smoke:", SMOKE)

## 1. Data: download, label mapping, sampling
AFND layout (per the dataset description): `Dataset/source_N/scraped_articles.json` (array of article objects) and `sources.json` (one line per source with its Misbar-based label).
The cell below prints what it finds; the label counts should match the paper: 52 credible / 51 not credible / 31 undecided sources.

In [ ]:
import kagglehub
DATA_ROOT = kagglehub.dataset_download("murtadhayaseen/arabic-fake-news-dataset-afnd")
# DATA_ROOT = "/content/drive/MyDrive/AFND"   # <- alternative: your own copy (e.g. from Mendeley)
print("data root:", DATA_ROOT)
for p in sorted(glob.glob(os.path.join(DATA_ROOT, "*")))[:10]: print("  ", p)

src_files = glob.glob(os.path.join(DATA_ROOT, "**", "sources.json"), recursive=True)
art_files = glob.glob(os.path.join(DATA_ROOT, "**", "scraped_articles.json"), recursive=True)
print("sources.json:", src_files, "| article files found:", len(art_files))
assert src_files and art_files, "Could not find sources.json / scraped_articles.json - check DATA_ROOT"
print("\nFirst 3 raw lines of sources.json:")
with open(src_files[0], encoding="utf-8") as f:
    for _ in range(3): print("  ", f.readline().strip()[:200])

In [ ]:
SRC_RE = re.compile(r"source[_ ]?(\d+)", re.I)

def norm_label(v):
    s = str(v).lower().replace("_", " ").strip()
    if "undecid" in s or "mixed" in s: return "U"
    if s.startswith("not") or "non" in s or "fake" in s or "unreliable" in s: return 0
    if "cred" in s or "real" in s or s == "true": return 1
    return None

def parse_sources(path):
    # Tolerant parser: finds a 'source_N' id and a credibility label in each record.
    txt = open(path, encoding="utf-8").read().strip()
    try: obj = json.loads(txt)
    except json.JSONDecodeError: obj = [json.loads(l) for l in txt.splitlines() if l.strip()]
    if isinstance(obj, dict):
        recs = [({"source": k, **v} if isinstance(v, dict) else {k: v}) for k, v in obj.items()]
    else:
        recs = obj
    out = {}
    for r in recs:
        pairs = list(r.items()) if isinstance(r, dict) else [(None, r)]
        sid = lab = None
        for k, v in pairs:
            for cand in (k, v):
                m = SRC_RE.fullmatch(str(cand).strip()) if cand is not None else None
                if m and sid is None: sid = f"source_{m.group(1)}"
        for k, v in pairs:
            if str(v) != str(sid) and lab is None and norm_label(v) is not None: lab = norm_label(v)
        if sid is not None and lab is not None: out[sid] = lab
    return out

src_label = parse_sources(src_files[0])
counts = pd.Series(src_label).map({1: "credible", 0: "not credible", "U": "undecided"}).value_counts()
print(counts.to_string())
print("expected from the AFND paper: credible 52 | not credible 51 | undecided 31")
assert len(src_label) > 100, "Label parsing failed - inspect the raw lines printed above and adapt parse_sources()"

In [ ]:
def clean(t):
    t = re.sub(r"https?://\S+|www\.\S+", " ", str(t))
    t = re.sub(r"[\u064B-\u0652\u0640]", "", t)      # tashkeel + tatweel
    return re.sub(r"\s+", " ", t).strip()

def load_articles(path):
    with open(path, encoding="utf-8") as f:
        raw = f.read().strip()
    try: arts = json.loads(raw)
    except json.JSONDecodeError: arts = [json.loads(l) for l in raw.splitlines() if l.strip()]
    return arts if isinstance(arts, list) else list(arts.values())

def flatten(arts):
    # AFND files may wrap articles in extra lists: unwrap until we reach the article objects
    out = []
    for x in arts:
        if isinstance(x, list) and x and all(isinstance(y, (list, dict)) for y in x): out.extend(flatten(x))
        else: out.append(x)
    return out

def to_article(x):
    # article as dict {title,text,date} OR as a list of strings (positional: title, text, date)
    if isinstance(x, dict):
        return x.get("title", ""), x.get("text", x.get("content", x.get("body", "")))
    if isinstance(x, (list, tuple)):
        strs = [t for t in x if isinstance(t, str)]
        if not strs: return "", ""
        body = max(strs, key=len)
        return (strs[0] if strs[0] is not body else ""), body
    return "", (x if isinstance(x, str) else "")

rng = np.random.RandomState(SEED)
seen, rows, per_src = set(), [], {}
first_sid = next(k for k, v in sorted(src_label.items()) if v in (0, 1))
t0 = time.time()
for sid, lab in sorted(src_label.items()):
    if lab not in (0, 1): continue                    # drop 'undecided'
    p = [f for f in art_files if os.path.basename(os.path.dirname(f)) == sid]
    if not p: continue
    arts = flatten(load_articles(p[0]))
    if not arts: continue
    if sid == first_sid: print("example article object:", type(arts[0]).__name__, "|", str(arts[0])[:300])
    order = rng.permutation(len(arts)); kept = 0
    for i in order:
        title, body = to_article(arts[i]); title, body = clean(title), clean(body)
        if len(body) < CFG["min_chars"]: continue
        text = (title + " . " + body)[:CFG["max_chars"]]
        h = hashlib.md5(text.encode("utf-8")).hexdigest()
        if h in seen: continue                        # global exact-duplicate removal
        seen.add(h); rows.append((sid, lab, text)); kept += 1
        if kept >= CFG["n_per_source"]: break
    per_src[sid] = (lab, len(arts), kept); del arts
df = pd.DataFrame(rows, columns=["source", "label", "text"]).reset_index(drop=True)
print(f"loaded in {time.time()-t0:.0f}s | articles: {len(df)} | sources: {df.source.nunique()}")
print("articles per class:", df.label.value_counts().to_dict(), "| sources per class:",
      df.groupby('label').source.nunique().to_dict())
df.to_csv("results/sample_afnd.csv", index=False)    # exact sample used in every experiment

## 2. Evaluation protocols
* **Protocol A - random split:** stratified article-level 70/15/15 (what prior AFND work effectively does).
* **Protocol B - source-disjoint split:** whole *sources* are assigned to train/val/test (stratified by class at the source level), so no website appears in two parts. Repeated for `n_repeats` different seeds.

In [ ]:
def random_split(df, seed):
    a, b, _ = CFG["split"]
    tr, tmp = train_test_split(np.arange(len(df)), test_size=1 - a, stratify=df.label, random_state=seed)
    va, te = train_test_split(tmp, test_size=0.5, stratify=df.label.values[tmp], random_state=seed)
    return np.asarray(tr), np.asarray(va), np.asarray(te)

def source_split(df, seed):
    a, b, _ = CFG["split"]; r = np.random.RandomState(seed); assign = {}
    for lab in (0, 1):                                  # stratify sources by class
        srcs = np.array(sorted(df.loc[df.label == lab, "source"].unique())); r.shuffle(srcs)
        n = len(srcs); i, j = int(round(a * n)), int(round((a + b) * n))
        for s in srcs[:i]: assign[s] = "train"
        for s in srcs[i:j]: assign[s] = "val"
        for s in srcs[j:]: assign[s] = "test"
    part = df.source.map(assign).values
    tr, va, te = (np.where(part == p)[0] for p in ("train", "val", "test"))
    S = lambda idx: set(df.source.values[idx])
    assert not (S(tr) & S(va)) and not (S(tr) & S(te)) and not (S(va) & S(te)), "source leakage!"
    return tr, va, te

def metrics(y, p):
    return dict(macro_f1=f1_score(y, p, average="macro"), acc=accuracy_score(y, p),
                recall_not_credible=recall_score(y, p, pos_label=0),
                recall_credible=recall_score(y, p, pos_label=1))

SPLITTERS = {"random": random_split, "source_disjoint": source_split}
tr, va, te = source_split(df, SEED)
print("source-disjoint sizes (train/val/test articles):", len(tr), len(va), len(te),
      "| test sources:", df.source.values[te].shape and len(set(df.source.values[te])))

## 3. Baseline 1 - TF-IDF + linear SVM

In [ ]:
def run_svm(df, tr, te, seed, return_model=False):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=300_000, sublinear_tf=True)
    Xtr = vec.fit_transform(df.text.values[tr]); Xte = vec.transform(df.text.values[te])
    clf = LinearSVC(C=1.0, random_state=seed).fit(Xtr, df.label.values[tr])
    m = metrics(df.label.values[te], clf.predict(Xte))
    return (m, vec, clf) if return_model else m

results = []
for r in range(CFG["n_repeats"]):
    seed = SEED + r
    for proto, fn in SPLITTERS.items():
        tr, va, te = fn(df, seed)
        m = run_svm(df, tr, te, seed)
        results.append(dict(model="tfidf_svm", protocol=proto, seed=seed, n_train=len(tr), n_test=len(te), **m))
        print(f"[SVM] seed {seed} {proto:16s} macro-F1 {m['macro_f1']:.4f} acc {m['acc']:.4f}")
pd.DataFrame(results).to_csv("results/results.csv", index=False)

## 4. Baseline 2 - AraBERT fine-tuning (default: uniform LR, unweighted loss)
LLRD (Eq. 1) and class-weighted loss are switched on through `xi` and `weighted`; the defaults reproduce standard fine-tuning.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
tok = AutoTokenizer.from_pretrained(CFG["model_name"])

def encode(texts):
    enc = tok(list(texts), truncation=True, max_length=CFG["max_len"], padding="max_length", return_tensors="pt")
    return enc["input_ids"], enc["attention_mask"]

def make_param_groups(model, lr_top, xi, wd):
    # Eq. (1): layer l gets lr_top * xi**(L-l). l=0 embeddings, l=1..L encoder layers, pooler+classifier use l=L.
    L = model.config.num_hidden_layers
    no_decay = ("bias", "LayerNorm.weight", "LayerNorm.bias")
    def layer_id(n):
        if n.startswith(("classifier", "bert.pooler")): return L
        if "bert.embeddings" in n: return 0
        return int(re.search(r"encoder\.layer\.(\d+)\.", n).group(1)) + 1
    named = list(model.named_parameters()); groups = []
    for l in range(L + 1):
        ps = [(n, p) for n, p in named if layer_id(n) == l]; lr = lr_top * xi ** (L - l)
        dec = [p for n, p in ps if not any(k in n for k in no_decay)]
        nod = [p for n, p in ps if any(k in n for k in no_decay)]
        if dec: groups.append({"params": dec, "lr": lr, "weight_decay": wd})
        if nod: groups.append({"params": nod, "lr": lr, "weight_decay": 0.0})
    assert sum(len(g["params"]) for g in groups) == len(named), "some parameters missing from groups"
    return groups

@torch.no_grad()
def predict(model, ids, mask, bs=128):
    model.eval(); out = []
    for i in range(0, len(ids), bs):
        with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
            lg = model(input_ids=ids[i:i+bs].to(DEVICE), attention_mask=mask[i:i+bs].to(DEVICE)).logits
        out.append(lg.argmax(-1).cpu())
    return torch.cat(out).numpy()

def train_eval_bert(df, tr, va, te, seed, xi=1.0, weighted=False, return_model=False, tag=""):
    set_seed(seed); t0 = time.time()
    Xtr, Mtr = encode(df.text.values[tr]); Xva, Mva = encode(df.text.values[va]); Xte, Mte = encode(df.text.values[te])
    ytr = torch.tensor(df.label.values[tr]); yva, yte = df.label.values[va], df.label.values[te]
    model = AutoModelForSequenceClassification.from_pretrained(CFG["model_name"], num_labels=2).to(DEVICE)
    w = None
    if weighted:                                        # w_c = N / (K * n_c)
        cnt = np.bincount(ytr.numpy(), minlength=2); w = torch.tensor(len(ytr) / (2 * cnt), dtype=torch.float, device=DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(make_param_groups(model, CFG["lr_top"], xi, CFG["weight_decay"]))
    bs = CFG["batch_size"]; steps = CFG["epochs"] * math.ceil(len(tr) / bs)
    sched = get_linear_schedule_with_warmup(opt, int(CFG["warmup"] * steps), steps)
    scaler = torch.cuda.amp.GradScaler(enabled=DEVICE == "cuda")
    g = torch.Generator().manual_seed(seed); best, best_state = -1, None
    for ep in range(CFG["epochs"]):
        model.train(); perm = torch.randperm(len(tr), generator=g)
        for i in range(0, len(tr), bs):
            b = perm[i:i+bs]
            with torch.autocast("cuda", dtype=torch.float16, enabled=DEVICE == "cuda"):
                lg = model(input_ids=Xtr[b].to(DEVICE), attention_mask=Mtr[b].to(DEVICE)).logits
            loss = loss_fn(lg.float(), ytr[b].to(DEVICE))
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); sched.step()
        f1 = f1_score(yva, predict(model, Xva, Mva), average="macro")
        print(f"   {tag} epoch {ep+1}/{CFG['epochs']} val macro-F1 {f1:.4f} ({time.time()-t0:.0f}s)")
        if f1 > best: best, best_state = f1, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    m = metrics(yte, predict(model, Xte, Mte)); m["val_macro_f1"] = best; m["train_seconds"] = time.time() - t0
    if return_model: return m, model
    del model, opt; torch.cuda.empty_cache(); return m

In [ ]:
bert_rows = []
for r in range(CFG["bert_repeats"]):
    seed = SEED + r
    for proto, fn in SPLITTERS.items():
        tr, va, te = fn(df, seed)
        m = train_eval_bert(df, tr, va, te, seed, xi=1.0, weighted=False, tag=f"[default/{proto}]")
        bert_rows.append(dict(model="arabert_default", protocol=proto, seed=seed, n_train=len(tr), n_test=len(te), **m))
        print(f"[AraBERT default] seed {seed} {proto:16s} macro-F1 {m['macro_f1']:.4f} acc {m['acc']:.4f}")
results += bert_rows
pd.DataFrame(results).to_csv("results/results.csv", index=False)

## 5. Baseline summary and the random vs. source-disjoint gap

In [ ]:
res = pd.DataFrame(results)
summ = res.groupby(["model", "protocol"])[["macro_f1", "acc", "recall_not_credible", "recall_credible"]].agg(["mean", "std"]).round(4)
display(summ)
piv = res.pivot_table(index="model", columns="protocol", values="macro_f1", aggfunc="mean")
piv["gap (random - source_disjoint)"] = piv["random"] - piv["source_disjoint"]
display(piv.round(4))
piv.to_csv("results/gap_table.csv"); summ.to_csv("results/summary.csv")

## 6. Remaining experiments (set `RUN_ABLATION = True` / `RUN_ANALYSIS = True` in the config cell)
Ablation: default, LLRD only, weighted loss only, both - all on the source-disjoint protocol.

In [ ]:
if RUN_ABLATION:
    ABL = {"default": (1.0, False), "llrd": (CFG["llrd_xi"], False),
           "weighted": (1.0, True), "llrd+weighted": (CFG["llrd_xi"], True)}
    abl_rows = []
    for r in range(CFG["bert_repeats"]):
        seed = SEED + r; tr, va, te = source_split(df, seed)
        for name, (xi, wt) in ABL.items():
            m = train_eval_bert(df, tr, va, te, seed, xi=xi, weighted=wt, tag=f"[{name}]")
            abl_rows.append(dict(config=name, seed=seed, **m)); print(name, round(m["macro_f1"], 4))
    abl = pd.DataFrame(abl_rows); abl.to_csv("results/ablation.csv", index=False)
    display(abl.groupby("config")[["macro_f1", "acc", "recall_not_credible", "recall_credible"]].agg(["mean", "std"]).round(4))

## 7. Analysis: what do the models rely on? (SHAP + feature weights)

In [ ]:
if RUN_ANALYSIS:
    # (a) SVM: strongest features under each protocol -> look for site names / boilerplate / topics
    for proto, fn in SPLITTERS.items():
        tr, va, te = fn(df, SEED)
        m, vec, clf = run_svm(df, tr, te, SEED, return_model=True)
        names = np.array(vec.get_feature_names_out()); order = np.argsort(clf.coef_[0])
        print(f"\n=== {proto} | macro-F1 {m['macro_f1']:.3f}")
        print("top NOT-credible features:", list(names[order[:25]]))
        print("top CREDIBLE features    :", list(names[order[-25:]]))
    # (b) SHAP on AraBERT (source-disjoint split, 20 test articles)
    import shap
    from transformers import pipeline
    tr, va, te = source_split(df, SEED)
    m, model = train_eval_bert(df, tr, va, te, SEED, return_model=True, tag="[shap model]")
    pipe = pipeline("text-classification", model=model, tokenizer=tok, device=0 if DEVICE == "cuda" else -1,
                    top_k=None, truncation=True, max_length=CFG["max_len"])
    texts = list(df.text.values[te][:20])
    sv = shap.Explainer(pipe)(texts)
    shap.plots.text(sv[:, :, 1])                      # class index 1 = credible

## 8. Reproducibility notes
* Seeds: `SEED = 42` (+ repeat index); all splits, sampling and training are seeded. GPU kernels can still make runs differ slightly.
* Outputs: `results/sample_afnd.csv` (exact sample), `results/results.csv`, `results/summary.csv`, `results/gap_table.csv`, `results/ablation.csv`.
* To reproduce the paper tables: set `SMOKE = False`, `bert_repeats = 3`, `RUN_ABLATION = RUN_ANALYSIS = True`, then *Run all*.